# 03 — Trends Over Time

Companion to [`../../data_visualization/trends.md`](../../data_visualization/trends.md).

---

## Why time series visualization matters

Time series data has a unique structure: each observation is tied to a specific point in time. This enables analyses that are impossible with cross-sectional data:

- **Trend**: Is the metric going up, down, or staying flat over the long term?
- **Seasonality**: Are there repeating patterns (daily, weekly, yearly)?
- **Anomalies**: Are there unexpected spikes or drops?
- **Autocorrelation**: Does today's value depend on yesterday's?

This notebook walks through the essential tools for exploring time series data.

All data used is drawn from the handbook's [`sample_data`](../sample_data/) directory with industry-relevant context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import acf

sns.set_theme(style='whitegrid', font_scale=1.0)
rng = np.random.default_rng(42)

print('Libraries loaded.')

## 1. Raw line plot — the foundation

Using `sample_data/trends/line_graph.csv` — **monthly product sales** over time.
This mirrors a **retail/e-commerce** scenario tracking product performance.

In [ ]:
line_data = pd.read_csv('../sample_data/trends/line_graph.csv')
line_data['month'] = pd.to_datetime(line_data['month'])
line_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

palette = {'Product A': '#2ecc71', 'Product B': '#3498db', 'Product C': '#e74c3c'}
for product, color in palette.items():
    subset = line_data[line_data['product'] == product]
    ax.plot(subset['month'], subset['sales'], color=color, linewidth=2, label=product, marker='o', markersize=4)

ax.set_xlabel('Month'); ax.set_ylabel('Sales')
ax.set_title('Monthly Product Sales — Line Graph\n(Retail/E-commerce Trend Tracking)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Area graph — showing volume over time

Using `sample_data/trends/area_graph.csv` — **website traffic by source**.
This is a **digital analytics** scenario: organic vs paid vs social traffic.

In [ ]:
area_data = pd.read_csv('../sample_data/trends/area_graph.csv')
area_data['date'] = pd.to_datetime(area_data['date'])
area_data.head()

In [ ]:
# Pivot for stacked area
pivoted = area_data.pivot_table(index='date', columns='traffic_source', values='visitors', aggfunc='sum')

fig, ax = plt.subplots(figsize=(14, 5))
pivoted.plot.area(ax=ax, alpha=0.5, colormap='Set2')
ax.set_xlabel('Date'); ax.set_ylabel('Visitors')
ax.set_title('Website Traffic by Source — Area Graph\n(Digital Analytics)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Smoothing — rolling means

A rolling (moving) average smooths out noise to reveal the underlying trend.

In [ ]:
# Generate a realistic time series with trend, seasonality, and noise
idx = pd.date_range('2022-01-01', '2024-12-31', freq='D')
trend = np.linspace(100, 200, len(idx))
weekly = 10 * np.sin(2 * np.pi * idx.dayofyear / 7)
yearly = 30 * np.sin(2 * np.pi * idx.dayofyear / 365)
noise = rng.normal(0, 8, len(idx))
ts_df = pd.DataFrame({'y': trend + weekly + yearly + noise}, index=idx)

fig, ax = plt.subplots(figsize=(14, 5))

# Raw data (light)
ax.plot(ts_df.index, ts_df['y'], color='lightgray', linewidth=0.5, label='daily')

# Multiple rolling windows
for window, color, label in [(7, 'orange', '7-day'), (30, 'red', '30-day'), (90, 'purple', '90-day')]:
    rolled = ts_df['y'].rolling(window, center=True).mean()
    ax.plot(ts_df.index, rolled, color=color, linewidth=1.5, label=f'{label} rolling mean')

ax.set_xlabel('Date'); ax.set_ylabel('y')
ax.set_title('Raw Series + Multiple Rolling Averages\n(Larger window = smoother trend)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.tight_layout(); plt.show()

## 4. STL decomposition

STL (Seasonal-Trend decomposition using LOESS) separates a time series into:
- **Trend**: The long-term direction
- **Seasonal**: The repeating short-term pattern
- **Residual**: What's left (noise + anomalies)

In [ ]:
import matplotlib
old_figsize = matplotlib.rcParams['figure.figsize']
matplotlib.rcParams['figure.figsize'] = (14, 7)

res = STL(ts_df['y'], period=365, robust=True).fit()
fig = res.plot()
fig.suptitle('STL Decomposition (period=365 days)\n(Trend + Seasonality + Residuals)', y=1.02)
matplotlib.rcParams['figure.figsize'] = old_figsize
plt.tight_layout(); plt.show()

## 5. Seasonal analysis

Understanding seasonality is key. Let's look at patterns by different time units.

In [ ]:
df_seasonal = ts_df.copy()
df_seasonal['month'] = df_seasonal.index.month
df_seasonal['dow'] = df_seasonal.index.dayofweek
df_seasonal['year'] = df_seasonal.index.year
df_seasonal['dayofyear'] = df_seasonal.index.dayofyear

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Monthly box plots
sns.boxplot(data=df_seasonal, x='month', y='y', ax=axes[0], palette='Set2')
axes[0].set_xticklabels(month_names)
axes[0].set_xlabel('Month'); axes[0].set_ylabel('y')
axes[0].set_title('Within-Year Seasonality\n(Monthly Pattern)')

# Day-of-week pattern
sns.boxplot(data=df_seasonal, x='dow', y='y', ax=axes[1], palette='Set2')
axes[1].set_xticklabels(dow_names)
axes[1].set_xlabel('Day of week'); axes[1].set_ylabel('y')
axes[1].set_title('Within-Week Seasonality\n(Day-of-Week Pattern)')

# Seasonal subseries — one line per year
for year, color in [(2022, '#2ecc71'), (2023, '#3498db'), (2024, '#e74c3c')]:
    subset = df_seasonal[df_seasonal['year'] == year].sort_values('dayofyear')
    axes[2].plot(subset['dayofyear'], subset['y'], color=color, alpha=0.7, label=str(year))
axes[2].set_xlabel('Day of year'); axes[2].set_ylabel('y')
axes[2].set_title('Seasonal Subseries\n(One line per year)')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 6. Autocorrelation analysis

Autocorrelation measures how much today's value correlates with yesterday's (lag-1), the day before (lag-2), etc.

In [ ]:
acf_values = acf(ts_df['y'], nlags=365, fft=True)

fig, ax = plt.subplots(figsize=(14, 4))
stem = ax.stem(acf_values[:120], linefmt='-', markerfmt='o', basefmt='black')
stem.stemlines.set_color('steelblue')
stem.markerline.set_color('steelblue')
ax.set_xlabel('Lag (days)'); ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation Function (ACF) — First 120 Days\n(Peaks at lag=7 = weekly seasonality)')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=2/np.sqrt(len(ts_df)), color='red', linestyle='--', alpha=0.5, label='95% confidence')
ax.axhline(y=-2/np.sqrt(len(ts_df)), color='red', linestyle='--', alpha=0.5)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Lag plot

A lag plot scatter-plots each point against its lagged value. A tight diagonal cluster = strong autocorrelation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for lag, ax in zip([1, 7, 365], axes):
    lagged = pd.DataFrame({'y_t': ts_df['y'].values[lag:], 'y_t-lag': ts_df['y'].values[:-lag]})
    ax.scatter(lagged['y_t-lag'], lagged['y_t'], alpha=0.05, s=5, color='steelblue')
    ax.set_xlabel(f'y(t - {lag})')
    ax.set_ylabel(f'y(t)')
    ax.set_title(f'Lag Plot (lag={lag})')
    corr = lagged.corr().iloc[0, 1]
    ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=10, 
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout(); plt.show()

## 8. Residual analysis

After removing trend and seasonality, the residuals should look like random noise.

In [ ]:
residuals = res.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuals over time
axes[0, 0].plot(residuals.index, residuals, color='steelblue', linewidth=0.5, alpha=0.7)
axes[0, 0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0, 0].set_xlabel('Date'); axes[0, 0].set_ylabel('Residual')
axes[0, 0].set_title('Residuals Over Time\n(Should look like random noise)')
axes[0, 0].grid(True, alpha=0.3)

# Residual histogram + KDE
sns.histplot(residuals, bins=50, ax=axes[0, 1], kde=True, color='steelblue', alpha=0.7)
axes[0, 1].set_xlabel('Residual'); axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title(f'Residual Distribution\n(mean={residuals.mean():.2f}, std={residuals.std():.2f})')

# Residual ACF
res_acf = acf(residuals, nlags=60, fft=True)
stem = axes[1, 0].stem(res_acf, linefmt='-', markerfmt='o', basefmt='black')
stem.stemlines.set_color('steelblue')
stem.markerline.set_color('steelblue')
axes[1, 0].set_xlabel('Lag'); axes[1, 0].set_ylabel('Autocorrelation')
axes[1, 0].set_title('Residual ACF\n(No significant peaks = good!)')
axes[1, 0].axhline(y=2/np.sqrt(len(residuals)), color='red', linestyle='--', alpha=0.5)
axes[1, 0].axhline(y=-2/np.sqrt(len(residuals)), color='red', linestyle='--', alpha=0.5)

# Residuals by month
df_res = pd.DataFrame({'residual': residuals}, index=residuals.index)
df_res['month'] = df_res.index.month
sns.boxplot(data=df_res, x='month', y='residual', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_xticklabels(month_names)
axes[1, 1].set_xlabel('Month'); axes[1, 1].set_ylabel('Residual')
axes[1, 1].set_title('Residuals by Month\n(No pattern = seasonality fully removed)')

plt.tight_layout(); plt.show()

## 9. Stacked area — cumulative total over time

Using `sample_data/trends/stacked_area.csv` — **user segments over time**.
This is a **SaaS/metrics** scenario: tracking how user segments evolve.

In [ ]:
stacked_data = pd.read_csv('../sample_data/trends/stacked_area.csv')
stacked_data['month'] = pd.to_datetime(stacked_data['month'])
stacked_data.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

pivot = stacked_data.pivot_table(index='month', columns='segment', values='users', aggfunc='sum')
pivot.plot.area(ax=ax, alpha=0.6, colormap='Set2')
ax.set_xlabel('Month'); ax.set_ylabel('Users')
ax.set_title('User Segments Over Time — Stacked Area\n(SaaS Metrics)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 10. Stream graph — organic flowing areas

Using `sample_data/trends/stream_graph.csv` — **feature engagement over time**.
A stream graph is a modified stacked area plot shifted around a center line.

In [ ]:
stream_data = pd.read_csv('../sample_data/trends/stream_graph.csv')
stream_data['date'] = pd.to_datetime(stream_data['date'])
stream_data.head()

In [ ]:
# Stream graph using plotly for the offset
import plotly.express as px

fig = px.area(stream_data, x='date', y='engagement', color='feature',
              title='Feature Engagement — Stream Graph\n(Web Analytics)',
              labels={'date': 'Date', 'engagement': 'Engagement', 'feature': 'Feature'},
              color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_layout(showlegend=True)
fig.show()

## Pitfalls

- **Don't use bar charts for time series** — they imply discrete categories, not continuous flow.
- **Avoid dual y-axes** — they make it hard to read both series accurately.
- **A truncated y-axis on a line chart is sometimes OK** — but mark it clearly with a break symbol.
- **Rolling averages create gaps** at the edges — be aware of the centering trade-off.
- **Autocorrelation ≠ causation** — a lagged variable can be correlated without being predictive.